In [1]:
!pip install roboflow ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 5.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.2 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_API_KEY")

In [3]:
from IPython.display import Image, display
from roboflow import Roboflow
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter
from ultralytics import YOLO
import random
import cv2
import numpy as np
import yaml
import json
import requests
from tqdm import tqdm


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## Dataset Selection

BDD100K was considered initially, but it does not provide the required combination of `cone`, `barrier`, and `stop_sign` objects in the construction/facility-navigation context targeted by this project.

The primary dataset is therefore `roadwork_VPanels_all_objects`, as its roadwork and construction scenes are closer to the intended operating environment.

From this dataset, the classes are mapped to the navigation task as follows:

- `Cone` → `cone`
- `Tubular Marker` → `cone`
- `Drum` → `cone`
- `Barrier` → `barrier`

Directional panels are excluded because they are not treated as navigation obstacles in this task.

In [4]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace("roadworkwithdirectionalpanelsallobjects").project("roadwork_vpanels_all_objects")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to roadwork_VPanels_all_objects-1 in yolov8:: 100%|██████████| 10685/10685 [00:01<00:00, 6720.97it/s]


In [5]:
with open('/kaggle/working/roadwork_VPanels_all_objects-1/data.yaml', 'r') as f:
    content = f.read()
    print(content)

names:
- Barrier
- Cone
- Drum
- Tubular Marker
- Vertical Panel
- Vertical Panel pass left
- Vertical Panel pass right
nc: 7
roboflow:
  license: CC BY 4.0
  project: roadwork_vpanels_all_objects
  url: https://universe.roboflow.com/roadworkwithdirectionalpanelsallobjects/roadwork_vpanels_all_objects/dataset/1
  version: 1
  workspace: roadworkwithdirectionalpanelsallobjects
test: ../test/images
train: ../train/images
val: ../valid/images



## Dataset Preparation

The original dataset provides only a training split, so the retained images are divided into train, validation, and test sets using an 80/10/10 split.

At this point the dataset contains `barrier` and `cone`. `stop_sign` is added separately in the following step because it is not sufficiently represented in the roadwork dataset.

In [6]:
REMAP = {0: 0, 1: 1, 2: 1, 3: 1}  
DISCARD = {4, 5, 6}                 

BASE      = Path("/kaggle/working/roadwork_VPanels_all_objects-1")
TRAIN_IMG = BASE / "train/images"
TRAIN_LBL = BASE / "train/labels"
OUT       = Path("/kaggle/working/dataset")

for split in ["train", "val", "test"]:
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "labels").mkdir(parents=True, exist_ok=True)

def remap_label_file(src_lbl, dst_lbl):
    kept_lines = []
    with open(src_lbl) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls = int(parts[0])
            if cls in DISCARD:
                continue
            parts[0] = str(REMAP[cls])
            kept_lines.append(" ".join(parts))
    # only write if at least one kept annotation remains
    if kept_lines:
        with open(dst_lbl, "w") as f:
            f.write("\n".join(kept_lines))
        return True
    return False

all_imgs = sorted(TRAIN_IMG.glob("*"))
valid_pairs = []

for img_path in all_imgs:
    lbl_path = TRAIN_LBL / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
    kept = []
    with open(lbl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls = int(line.split()[0])
            if cls not in DISCARD:
                kept.append(line)
    if kept:
        valid_pairs.append((img_path, lbl_path))

print(f"Valid pairs after discard: {len(valid_pairs)}")

# split 80/10/10 
train_pairs, temp = train_test_split(valid_pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(temp, test_size=0.5, random_state=42)

print(f"train: {len(train_pairs)} | val: {len(val_pairs)} | test: {len(test_pairs)}")

def write_split(pairs, split_name):
    for img_path, lbl_path in pairs:
        dst_img = OUT / split_name / "images" / img_path.name
        dst_lbl = OUT / split_name / "labels" / (img_path.stem + ".txt")
        shutil.copy(img_path, dst_img)
        remap_label_file(lbl_path, dst_lbl)

write_split(train_pairs, "train")
write_split(val_pairs,   "val")
write_split(test_pairs,  "test")

print("Done. Dataset written to:", OUT)

# write new data.yaml 
yaml_content = f"""
train: {OUT}/train/images
val:   {OUT}/val/images
test:  {OUT}/test/images

nc: 2
names: ['barrier', 'cone']
"""

with open(OUT / "data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("data.yaml written.")

Valid pairs after discard: 4792
train: 3833 | val: 479 | test: 480
Done. Dataset written to: /kaggle/working/dataset
data.yaml written.


In [7]:
with open('/kaggle/working/dataset/data.yaml', 'r') as f:
    content = f.read()
    print(content)

train: /kaggle/working/dataset/train/images
val:   /kaggle/working/dataset/val/images
test:  /kaggle/working/dataset/test/images

nc: 2
names: ['barrier', 'cone']


In [8]:
dtrain = Path('/kaggle/working/dataset/train/labels')
class_counts = Counter()
def check_labels(path):
    for f in path.glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                class_counts[int(line.split()[0])] += 1
    return class_counts
print(check_labels(dtrain))


Counter({1: 26011, 0: 495})


In [9]:
def display_image_count():   
    dataset_root = Path("/kaggle/working/dataset")
    splits = ["train", "val", "test"]
    
    for split in splits:
        lbl_dir = dataset_root / split / "labels"
        class_counts = Counter()
        img_count = 0
    
        for f in lbl_dir.glob("*.txt"):
            img_count += 1
            for line in f.read_text().splitlines():
                if line.strip():
                    class_counts[int(line.split()[0])] += 1
    
        print(f"\n{split}:")
        print(f"  images : {img_count}")
        print(f"  barrier    (0): {class_counts[0]}")
        print(f"  cone        (1): {class_counts[1]}")
        print(f"  stop_sign   (2): {class_counts[2]}")
        print(f"  total annotations: {sum(class_counts.values())}")

In [10]:
lbl_dir = Path("/kaggle/working/dataset/train/labels")
img_dir = Path("/kaggle/working/dataset/train/images")

barrier_pairs = []
for f in lbl_dir.glob("*.txt"):
    classes = [int(l.split()[0]) for l in f.read_text().splitlines() if l.strip()]
    if 0 in classes:
        for ext in [".jpg", ".jpeg", ".png"]:
            img = img_dir / (f.stem + ext)
            if img.exists():
                barrier_pairs.append((img, f))
                break

print(f"Barrier pairs found: {len(barrier_pairs)}")

def augment_image(img):
    # flip horizontal
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    # brightness shift
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,2] *= random.uniform(0.6, 1.4)
    hsv[:,:,2]  = np.clip(hsv[:,:,2], 0, 255)
    img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    if random.random() > 0.5:
        img = cv2.GaussianBlur(img, (3, 3), 0)
    h, w  = img.shape[:2]
    angle = random.uniform(-10, 10)
    M     = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img   = cv2.warpAffine(img, M, (w, h),
                           borderMode=cv2.BORDER_REFLECT)
    return img

def flip_labels(lbl_path):
    """mirror x-center for horizontal flip — only called when flip was applied"""
    lines = []
    for line in lbl_path.read_text().splitlines():
        if not line.strip():
            continue
        parts   = line.split()
        parts[1] = str(round(1.0 - float(parts[1]), 6))
        lines.append(" ".join(parts))
    return lines

REPEAT = 10
for i in range(1, REPEAT + 1):
    for img_path, lbl_path in barrier_pairs:
        img       = cv2.imread(str(img_path))
        flipped   = random.random() > 0.5
        if flipped:
            img = cv2.flip(img, 1)
        # apply remaining augmentations
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,2] *= random.uniform(0.6, 1.4)
        hsv[:,:,2]  = np.clip(hsv[:,:,2], 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        if random.random() > 0.5:
            img = cv2.GaussianBlur(img, (3, 3), 0)
        h, w   = img.shape[:2]
        angle  = random.uniform(-10, 10)
        M      = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        img    = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

        new_stem = f"{img_path.stem}_aug{i}"
        cv2.imwrite(str(img_dir / (new_stem + img_path.suffix)), img)

        # fix labels if flipped
        if flipped:
            lines = flip_labels(lbl_path)
        else:
            lines = [l for l in lbl_path.read_text().splitlines() if l.strip()]

        (lbl_dir / (new_stem + ".txt")).write_text("\n".join(lines))

print(f"After augmented oversampling: {len(list(lbl_dir.glob('*.txt')))} train labels")

Barrier pairs found: 129
After augmented oversampling: 5123 train labels


## Stop-Sign Pseudo-Labels

The roadwork datasets do not provide sufficient stop-sign annotations. Since YOLOv8s is already pretrained on COCO, where `stop_sign` is an existing class, the COCO-pretrained model is used to identify stop signs in the training images.

Detections for COCO class `11` with confidence ≥ 0.5 are converted into YOLO annotations and added as `stop_sign` labels.

This provides an initial source of stop-sign supervision without treating visible stop signs as background during training.

In [11]:
STOP_SIGN_COCO_ID = 11
CONFIDENCE_THRESH = 0.6  

model = YOLO("yolov8s.pt")  

splits = ["train", "val", "test"]
dataset_root = Path("/kaggle/working/dataset")

stop_sign_counts = {"train": 0, "val": 0, "test": 0}

for split in splits:
    img_dir = dataset_root / split / "images"
    lbl_dir = dataset_root / split / "labels"

    for img_path in img_dir.glob("*"):
        results = model.predict(
            source    = str(img_path),
            conf      = CONFIDENCE_THRESH,
            classes   = [STOP_SIGN_COCO_ID],  
            verbose   = False
        )

        detections = results[0].boxes
        if detections is None or len(detections) == 0:
            continue

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        img_h, img_w = results[0].orig_shape

        new_lines = []
        for box in detections:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            # normalize to YOLO format
            x_center = ((x1 + x2) / 2) / img_w
            y_center = ((y1 + y2) / 2) / img_h
            width    = (x2 - x1) / img_w
            height   = (y2 - y1) / img_h
            new_lines.append(f"2 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        if new_lines:
            existing = lbl_path.read_text() if lbl_path.exists() else ""
            with open(lbl_path, "a") as f:
                if existing and not existing.endswith("\n"):
                    f.write("\n")
                f.write("\n".join(new_lines))
            stop_sign_counts[split] += len(new_lines)

print("Stop signs added per split:")
for split, count in stop_sign_counts.items():
    print(f"  {split}: {count}")

Stop signs added per split:
  train: 288
  val: 23
  test: 22


In [12]:
OUT = Path("/kaggle/working/dataset")

data = {
    "train": str(OUT / "train/images"),
    "val":   str(OUT / "val/images"),
    "test":  str(OUT / "test/images"),
    "nc":    3,
    "names": ["barrier", "cone", "stop_sign"]
}

with open(OUT / "data.yaml", "w") as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(open(OUT / "data.yaml").read())


train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images
nc: 3
names:
- barrier
- cone
- stop_sign



In [13]:
display_image_count()


train:
  images : 5123
  barrier    (0): 5445
  cone        (1): 34541
  stop_sign   (2): 288
  total annotations: 40274

val:
  images : 479
  barrier    (0): 67
  cone        (1): 3165
  stop_sign   (2): 23
  total annotations: 3255

test:
  images : 480
  barrier    (0): 49
  cone        (1): 3357
  stop_sign   (2): 22
  total annotations: 3428


Three training stages were evaluated to understand how additional barrier data and fine-tuning affect the detector.

- **v1** → baseline training on the initial dataset
- **v2** → fine-tuning after adding the additional barrier data
- **v3** → final training on the complete merged dataset

The training cells are kept commented so that opening the notebook does not trigger expensive training runs. Each cell contains the configuration used for that experiment and can be uncommented when reproduction is required.

In [14]:
'''
model = YOLO("yolov8s.pt")

model.train(
    data       = "/kaggle/working/dataset/data.yaml",
    epochs     = 50,
    imgsz      = 640,
    batch      = 16,
    patience   = 10,        # early stopping if val mAP stalls

    # optimizer
    optimizer  = "AdamW",
    lr0        = 0.001,
    lrf        = 0.01,      # final lr = lr0 * lrf
    warmup_epochs = 3,

    # loss weights — barrier gets higher penalty
    cls        = 2.0,       # classification loss weight
    box        = 7.5,       # box regression loss weight

    # augmentation
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    fliplr     = 0.5,
    flipud     = 0.0,       # disabled — robots don't see upside down
    mosaic     = 1.0,       # keep on — critical for small object detection
    mixup      = 0.1,
    shear      = 0.0,       # disabled — unrealistic distortion

    # saving
    project    = "/kaggle/working/runs",
    name       = "eric_navigation_v1",
    save       = True,
    plots      = True,
)'''

'\nmodel = YOLO("yolov8s.pt")\n\nmodel.train(\n    data       = "/kaggle/working/dataset/data.yaml",\n    epochs     = 50,\n    imgsz      = 640,\n    batch      = 16,\n    patience   = 10,        # early stopping if val mAP stalls\n\n    # optimizer\n    optimizer  = "AdamW",\n    lr0        = 0.001,\n    lrf        = 0.01,      # final lr = lr0 * lrf\n    warmup_epochs = 3,\n\n    # loss weights — barrier gets higher penalty\n    cls        = 2.0,       # classification loss weight\n    box        = 7.5,       # box regression loss weight\n\n    # augmentation\n    hsv_h      = 0.015,\n    hsv_s      = 0.7,\n    hsv_v      = 0.4,\n    degrees    = 10.0,\n    translate  = 0.1,\n    scale      = 0.5,\n    fliplr     = 0.5,\n    flipud     = 0.0,       # disabled — robots don\'t see upside down\n    mosaic     = 1.0,       # keep on — critical for small object detection\n    mixup      = 0.1,\n    shear      = 0.0,       # disabled — unrealistic distortion\n\n    # saving\n    project  

## Additional Barrier Dataset

A second dataset, `vulture-40`, is introduced specifically to improve barrier representation.

Unlike the original roadwork dataset, it contains barriers with different appearances, including concrete and yellow/black barriers. This adds visual variation that is missing from the original barrier samples.

The segmentation annotations are converted to bounding boxes, and the relevant barrier variants are mapped to the existing `barrier` class. Fence annotations are excluded because they are outside the target object definition.

In [15]:
print(os.listdir("/kaggle/working/"))

ndjson_path = "/kaggle/input/datasets/tarunpandianm/vulture-40-barrier/barrier.ndjson"  # fix path

with open(ndjson_path) as f:
    for i in range(2):
        first = json.loads(f.readline())
        print(json.dumps(first, indent=4))

['roadwork_VPanels_all_objects-1', 'yolov8s.pt', 'dataset', '.virtual_documents']
{
    "type": "dataset",
    "task": "segment",
    "name": "barrier",
    "description": "Street-level traffic scenes featuring various road safety equipment, including concrete barriers and black and yellow barriers, annotated for segmentation.",
    "bytes": 67917807,
    "url": "https://platform.ultralytics.com/vulture-40/datasets/barrier",
    "class_names": {
        "0": "barrier",
        "1": "black and yellowbarrier",
        "2": "concretebarrier",
        "3": "fence"
    },
    "version": "latest",
    "created_at": "2026-09-11T13:00:22.041Z",
    "updated_at": "2026-09-11T13:00:22.041Z"
}
{
    "type": "image",
    "file": "yt-qnHMDmhzDgI-0094_jpg.rf.pRHORvkWl2rI6HFAHUoH.jpg",
    "url": "https://cdn.ul.run/ap/i/01b92bbf5aae91c472a9ebc6fc39c792.jpg?Expires=1790654639&KeyName=key-v1&Signature=KDGq2rW5MSc5JSlIKMiUEbQoWp0",
    "width": 360,
    "height": 360,
    "split": "test",
    "annotati

In [16]:
NDJSON_PATH = "/kaggle/input/datasets/tarunpandianm/vulture-40-barrier/barrier.ndjson" 
OUT         = Path("/kaggle/working/barrier_extra")

for split in ["train", "val", "test"]:
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "labels").mkdir(parents=True, exist_ok=True)

REMAP   = {0: 0, 1: 0, 2: 0}
DISCARD = {3}

converted = {"train": 0, "val": 0, "test": 0}
skipped   = 0

with open(NDJSON_PATH) as f:
    for line in f:
        record = json.loads(line)

        if record["type"] != "image":
            continue

        # skip if no annotations key at all
        if "annotations" not in record:
            skipped += 1
            continue

        split    = record.get("split", "train")
        filename = record["file"]
        segments = record["annotations"].get("segments", [])

        if not segments:
            skipped += 1
            continue

        yolo_lines = []
        for seg in segments:
            cls_id = int(seg[0])
            if cls_id in DISCARD:
                continue

            new_cls = REMAP.get(cls_id)
            if new_cls is None:
                continue

            coords   = seg[1:]
            xs       = coords[0::2]
            ys       = coords[1::2]
            x_center = (min(xs) + max(xs)) / 2
            y_center = (min(ys) + max(ys)) / 2
            bw       = max(xs) - min(xs)
            bh       = max(ys) - min(ys)

            yolo_lines.append(
                f"{new_cls} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}"
            )

        if not yolo_lines:
            skipped += 1
            continue

        stem     = Path(filename).stem
        lbl_path = OUT / split / "labels" / (stem + ".txt")
        lbl_path.write_text("\n".join(yolo_lines))
        converted[split] += 1

print(f"Converted: {converted}")
print(f"Skipped: {skipped}")

Converted: {'train': 421, 'val': 121, 'test': 61}
Skipped: 4


In [17]:
failed = []

with open(NDJSON_PATH) as f:
    lines = f.readlines()

image_records = [json.loads(l) for l in lines if json.loads(l)["type"] == "image"]

for record in tqdm(image_records):
    if "annotations" not in record:
        continue

    split    = record.get("split", "train")
    filename = record["file"]
    url      = record["url"]

    stem     = Path(filename).stem
    lbl_path = OUT / split / "labels" / (stem + ".txt")

    # only download if we have a matching label file
    if not lbl_path.exists():
        continue

    # find extension from filename
    ext      = Path(filename).suffix or ".jpg"
    img_path = OUT / split / "images" / (stem + ext)

    if img_path.exists():
        continue

    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            img_path.write_bytes(resp.content)
        else:
            failed.append((filename, resp.status_code))
    except Exception as e:
        failed.append((filename, str(e)))

print(f"Failed downloads: {len(failed)}")
if failed:
    print(failed[:5])

# verify counts
for split in ["train", "val", "test"]:
    imgs = list((OUT / split / "images").glob("*"))
    lbls = list((OUT / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")

100%|██████████| 607/607 [04:13<00:00,  2.40it/s]

Failed downloads: 0
train: 421 images, 421 labels
val: 121 images, 121 labels
test: 61 images, 61 labels


In [18]:
import yaml
from pathlib import Path

OUT = Path("/kaggle/working/barrier_extra")

data = {
    "train": str(OUT / "train/images"),
    "val":   str(OUT / "val/images"),
    "test":  str(OUT / "test/images"),
    "nc":    1,
    "names": ["barrier"]
}

with open(OUT / "data.yaml", "w") as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(open(OUT / "data.yaml").read())

train: /kaggle/working/barrier_extra/train/images
val: /kaggle/working/barrier_extra/val/images
test: /kaggle/working/barrier_extra/test/images
nc: 1
names:
- barrier



In [19]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/working/barrier_extra")
DST = Path("/kaggle/working/dataset")

for split in ["train", "val", "test"]:
    src_imgs = list((SRC / split / "images").glob("*"))
    src_lbls = list((SRC / split / "labels").glob("*.txt"))

    for img in src_imgs:
        shutil.copy(img, DST / split / "images" / img.name)

    for lbl in src_lbls:
        shutil.copy(lbl, DST / split / "labels" / lbl.name)

    print(f"{split}: copied {len(src_imgs)} images, {len(src_lbls)} labels")

train: copied 421 images, 421 labels
val: copied 121 images, 121 labels
test: copied 61 images, 61 labels


In [20]:
'''
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/eric_navigation_v1/weights/best.pt")

model.train(
    data          = "/kaggle/working/dataset/data.yaml",
    epochs        = 30,
    imgsz         = 640,
    batch         = 16,
    patience      = 7,
    optimizer     = "AdamW",
    lr0           = 0.0001,    # low lr — continuing from v1
    lrf           = 0.01,
    warmup_epochs = 2,
    freeze        = 10,        # freeze backbone, retrain head on all 3 classes
    cls           = 2.0,
    box           = 7.5,
    hsv_h         = 0.01,
    hsv_s         = 0.5,
    hsv_v         = 0.4,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.3,
    fliplr        = 0.5,
    flipud        = 0.0,
    mosaic        = 1.0,
    mixup         = 0.05,
    shear         = 0.0,
    project       = "/kaggle/working/runs",
    name          = "eric_navigation_v2",
    save          = True,
    plots         = True,
)'''

'\nfrom ultralytics import YOLO\n\nmodel = YOLO("/kaggle/working/runs/eric_navigation_v1/weights/best.pt")\n\nmodel.train(\n    data          = "/kaggle/working/dataset/data.yaml",\n    epochs        = 30,\n    imgsz         = 640,\n    batch         = 16,\n    patience      = 7,\n    optimizer     = "AdamW",\n    lr0           = 0.0001,    # low lr — continuing from v1\n    lrf           = 0.01,\n    warmup_epochs = 2,\n    freeze        = 10,        # freeze backbone, retrain head on all 3 classes\n    cls           = 2.0,\n    box           = 7.5,\n    hsv_h         = 0.01,\n    hsv_s         = 0.5,\n    hsv_v         = 0.4,\n    degrees       = 5.0,\n    translate     = 0.1,\n    scale         = 0.3,\n    fliplr        = 0.5,\n    flipud        = 0.0,\n    mosaic        = 1.0,\n    mixup         = 0.05,\n    shear         = 0.0,\n    project       = "/kaggle/working/runs",\n    name          = "eric_navigation_v2",\n    save          = True,\n    plots         = True,\n)'

In [21]:
display_image_count()


train:
  images : 5544
  barrier    (0): 6294
  cone        (1): 34541
  stop_sign   (2): 288
  total annotations: 41123

val:
  images : 600
  barrier    (0): 310
  cone        (1): 3165
  stop_sign   (2): 23
  total annotations: 3498

test:
  images : 541
  barrier    (0): 175
  cone        (1): 3357
  stop_sign   (2): 22
  total annotations: 3554


In [23]:
with open('/kaggle/working/dataset/data.yaml', 'r') as f:
    print(f.read())

train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images
nc: 3
names:
- barrier
- cone
- stop_sign



## Final Dataset

After combining the original roadwork data, the additional barrier data, and the generated stop-sign annotations, the version 3 was training on the entire combined dataset, the training ontology is:

```text
0 → barrier
1 → cone
2 → stop_sign
```



In [ ]:
'''
model = YOLO("yolov8s.pt")

model.train(
    data       = "/kaggle/working/dataset/data.yaml",
    epochs     = 50,
    imgsz      = 640,
    batch      = 16,
    patience   = 10,        # early stopping if val mAP stalls

    # optimizer
    optimizer  = "AdamW",
    lr0        = 0.001,
    lrf        = 0.01,      # final lr = lr0 * lrf
    warmup_epochs = 3,

    # loss weights — barrier gets higher penalty
    cls        = 2.0,       # classification loss weight
    box        = 7.5,       # box regression loss weight

    # augmentation
    hsv_h      = 0.015,
    hsv_s      = 0.5,
    hsv_v      = 0.4,
    degrees    = 5.0,
    translate  = 0.1,
    scale      = 0.3,
    fliplr     = 0.5,
    flipud     = 0.0,       # disabled — robots don't see upside down
    mosaic     = 1.0,       # keep on — critical for small object detection
    mixup      = 0.1,
    shear      = 0.0,       # disabled — unrealistic distortion

    # saving
    project    = "/kaggle/working/runs",
    name       = "eric_navigation_v3",
    save       = True,
    plots      = True,
)'''